<br>

# ANÁLISIS DISCRIMINANTE CUADRÁTICO Y LINEAL

### QDA
Clasificador probab. model a ccad clase como distr- gaussiana con su propia matriz de covariancas, generando limites de decision cuadráticos



In [ ]:
"""
PARA EL EXAMEN:
    SABER QUE ES EL REG-PARAM, LA TOLERANCIA
                    reg_param -> interpola matrixz varianzas con identidad, regularizar la clase
                    tolerancia(LDA) -> regular/suavizar las diferencias entre clases (repartir)


    SABER UTILIZAR QDA / LDA EN UN EXPERIMENTO
        QDA y LDA son clasificadores generativos. Los clasificadores generativos expresan posteriors en funcion de priors y 
        densidades de las clases

        Priors: probabilidad de pertenecer a una clase antes de ver los datos del punto a clasificar
        Posteriors: probabilidad de pertenecer a una clase después de observar sus características
"""

'\nPARA EL EXAMEN:\n    SABER QUE ES EL REG-PARAM, LA TOLERANCIA\n                    reg_param -> interpola matrixz varianzas con identidad, regularizar la clase\n                    tolerancia(LDA) -> regular/suavizar las diferencias entre clases (repartir)\n\n\n    SABER UTILIZAR QDA / LDA EN UN EXPERIMENTO\n'

<np>

# QDA: Prueba con MNIST

In [ ]:
import numpy as np
import datasets
from sklearn.decomposition import PCA

ds = datasets.load_dataset("ylecun/mnist").with_format("numpy")
X_train = ds["train"][:]["image"].astype(np.float32).reshape(-1, 28 * 28) / 255.0
y_train = ds["train"][:]["label"].astype(np.uint8)
X_test = ds["test"][:]["image"].astype(np.float32).reshape(-1, 28 * 28) / 255.0
y_test = ds["test"][:]["label"].astype(np.uint8)
K = 400
pca = PCA(n_components=K)            # pca = PCA(n_components = K).fit(X_train)
X_train = pca.fit_transform(X_train) # X_train = pca.transform(X_train)
X_test = pca.transform(X_test)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

# X_train.shape: 60000 x 400 parámetros
# y_train.shape: 60000 clases
# X_test.shape: 10000 x 400 parámetros
# y_test.shape: 10000 clases

/home/adria/Documentos/GitHub/University/3rd_Year/PER/env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(60000, 400) (60000,) (10000, 400) (10000,)


In [3]:
# loguniform: distribución que seguiremos para muestrear valores de reg_param

from scipy.stats import loguniform; loguniform(1e-4, 1e-1).rvs(5)

array([0.05980695, 0.00065173, 0.00428723, 0.00598984, 0.01059492])

In [ ]:
# Precisión del test: tras aprender reg_param y ajustar QDA con el valor aprendido
# En QDA cada clase tiene su media y matriz de covarianzas. 

from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.model_selection import train_test_split, ShuffleSplit, RandomizedSearchCV
import time
start = time.time()

# BÚSQUEDA DE HIPERPARÁMETROS
clf = QuadraticDiscriminantAnalysis()
G = {'reg_param': loguniform(1e-4, 1e-1)}       # reg_param aplica una regularización a cada matriz de covarianzas --> estabiliza la covarianza
# reg_param = 0 --> sin regularización      reg_param pequeño --> estabiliza la covarianza sin deformarla mucho         reg_param grande --> empuja la covarianza hacia la Identidad, mas simple, menos varianza
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23)
RS = RandomizedSearchCV(clf, G, n_iter=10, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=32)
acc = RS.fit(X_train, y_train).score(X_test, y_test)

print(f"Precisión: {acc:.2%} con {RS.best_params_['reg_param']:.4f}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

Precisión: 95.97% con 0.0633
Tiempo (hh:mm:ss): 00:00:07


<np>

# LDA con MNIST

In [ ]:
# LDA: una media por clase y una matriz de covarianzas comun para todas
# Parametros (afectan a la estabilidad, regularizacion,etc, del modelo)
#       tol
#       shrinkage
#       solver
# NOSOTROS ELEGIMOS ESTOS PARÁMETROS


import numpy as np
import datasets
from sklearn.decomposition import PCA
# Importamos la dataset MNIST
ds = datasets.load_dataset("ylecun/mnist").with_format("numpy")
X_train = ds["train"][:]["image"].astype(np.float32).reshape(-1, 28 * 28) / 255.0
y_train = ds["train"][:]["label"].astype(np.uint8)
X_test = ds["test"][:]["image"].astype(np.float32).reshape(-1, 28 * 28) / 255.0
y_test = ds["test"][:]["label"].astype(np.uint8)
K = 400
pca = PCA(n_components=K)   # Realizamos la compresión PCA a K dimensiones:
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(60000, 400) (60000,) (10000, 400) (10000,)


In [ ]:
# TOLERANCIA tol: umbral de tolerancia numerica que controla cuando el algoritmo 
# considera que una matriz de covarianza es casi singular y debe aplicar una correccion interna.
# Suavizado de varianzas y covarianzas

# reg_param --> QDA: modifica covarianza
# tol --> LDA: decide cuando un autovalor es demasiado pequeño para ser fiable

In [15]:
# Precisón en test: tras aprender tol y ajustar LDA con el valor aprendido

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, ShuffleSplit, RandomizedSearchCV
from scipy.stats import loguniform
import time

start = time.time()
# BÚSQUEDA DE HIPERPARÁMETROS !!!!
clf = LinearDiscriminantAnalysis() # lDA base
G = {'tol': loguniform(1e-9, 1e-1)} # Definimos el espacio de búsqueda: se prueban distintos valores entre 1e-9 y 1e-1
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23) # Crea 2 particiones aleatorias del conjunto de entrenamiento
RS = RandomizedSearchCV(clf, G, n_iter=10, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=8) 
# búsqueda aleatoria del mejor tol para LDA:
#       Prueba 10 valores aleatorios de tol entre 1e-9 y 1e-1 (G)
#       Evalua cada uno con 2 validaciones cruzadas 
#       Usa accuracy como la métrica
#       Parareliza todo el trabajo
acc = RS.fit(X_train, y_train).score(X_test, y_test)# AQUÍ SE REALIZA LA BÚSQUEDA DE HIPERPARÁMETROS: devuelve el modelo RS ya entrenado, con quien hacemos score()

print(f"Precisión: {acc:.2%} con {RS.best_params_['tol']:.4f}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

Precisión: 87.30% con 0.0000
Tiempo (hh:mm:ss): 00:00:05


<np>

# EJERCICIOS

In [ ]:
# Ejercicios:
# FASHION-MNIST (K = 600)

# Normalizamos las imágenes y apliamos PCA para reducir la dim de 784 a K componentes principales
import numpy as np
import datasets
from sklearn.decomposition import PCA

ds = datasets.load_dataset("zalando-datasets/fashion_mnist").with_format("numpy")
X_train = ds["train"][:]["image"].astype(np.float32).reshape(-1, 28 * 28) / 255.0
y_train = ds["train"][:]["label"].astype(np.uint8)
X_test = ds["test"][:]["image"].astype(np.float32).reshape(-1, 28 * 28) / 255.0
y_test = ds["test"][:]["label"].astype(np.uint8)

K = 600
pca = PCA(n_components=K)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(60000, 600) (60000,) (10000, 600) (10000,)


In [8]:
# Entrenamos un clasificador QDA con búsqueda aleatoria de hiperparámetros sobre los datos PCA de la Bda


from scipy.stats import loguniform; loguniform(1e-4, 1e-1).rvs(5)
# QDA: Clasificador prob. que asume clases gaussianas con covarianzas diferentes por clase
# Importa validación cruzada aleatoria y búsqueda de hiperparámetros
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.model_selection import train_test_split, ShuffleSplit, RandomizedSearchCV
import time; start = time.time()

# BÚSQUEDA DE HIPERPARÁMETROS
clf = QuadraticDiscriminantAnalysis()
G = {'reg_param': loguniform(1e-4, 1e-1)} # reg_param: parámetro de regularizacion. Evita matrices de covarianzas singulares
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23) # 
RS = RandomizedSearchCV(clf, G, n_iter=10, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=32) # Prueba n_iter combinaciones aleatorias en paralelo (n_jobs = -1)

acc = RS.fit(X_train, y_train).score(X_test, y_test)

print(f"Precisión: {acc:.2%} con {RS.best_params_['reg_param']:.4f}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

Precisión: 76.58% con 0.0823
Tiempo (hh:mm:ss): 00:06:12


In [ ]:
# Entrenamos un clasificador

import numpy as np
import datasets
from sklearn.decomposition import PCA

ds = datasets.load_dataset("zalando-datasets/fashion_mnist").with_format("numpy")
X_train = ds["train"][:]["image"].astype(np.float32).reshape(-1, 28 * 28) / 255.0
y_train = ds["train"][:]["label"].astype(np.uint8)
X_test = ds["test"][:]["image"].astype(np.float32).reshape(-1, 28 * 28) / 255.0
y_test = ds["test"][:]["label"].astype(np.uint8)

K = 600
pca = PCA(n_components=K)
X_train = pca.fit_transform(X_train)  # primero calcula la matriz de covarianzas
X_test = pca.transform(X_test)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(60000, 600) (60000,) (10000, 600) (10000,)


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, ShuffleSplit, RandomizedSearchCV
from scipy.stats import loguniform
import time

start = time.time()
clf = LinearDiscriminantAnalysis()
G = {"tol": loguniform(1e-9, 1e-1)}
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23)
RS = RandomizedSearchCV(
    clf, G, n_iter=10, scoring="accuracy", n_jobs=-1, cv=splitter, pre_dispatch=8
)
acc = RS.fit(X_train, y_train).score(X_test, y_test)
print(f"Precisión: {acc:.2%} con {RS.best_params_['tol']:.4f}")
print("Tiempo (hh:mm:ss):", time.strftime("%H:%M:%S", time.gmtime(time.time() - start)))

Precisión: 81.59% con 0.0019
Tiempo (hh:mm:ss): 00:00:26


In [ ]:
# Ejercicios: 
# CIFAR-10 (K = 600)

# Normalizamos las imágenes y apliamos PCA para reducir la dim de 784 a K componentes principales
import numpy as np; import datasets; from sklearn.decomposition import PCA
ds = datasets.load_dataset("uoft-cs/cifar10").with_format("numpy")
X_train = ds['train'][:]['img'].astype(np.float32).reshape(-1, 32*32*3) / 255.
y_train = ds['train'][:]['label'].astype(np.uint8)
X_test = ds['test'][:]['img'].astype(np.float32).reshape(-1, 32*32*3) / 255.
y_test = ds['test'][:]['label'].astype(np.uint8)

K = 600; pca = PCA(n_components=K); X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(50000, 600) (50000,) (10000, 600) (10000,)


In [12]:
# Entrenamos un clasificador QDA con búsqueda aleatoria de hiperparámetros sobre los datos PCA de la Bda


from scipy.stats import loguniform; loguniform(1e-4, 1e-1).rvs(5)
# QDA: Clasificador prob. que asume clases gaussianas con covarianzas diferentes por clase
# Importa validación cruzada aleatoria y búsqueda de hiperparámetros
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.model_selection import train_test_split, ShuffleSplit, RandomizedSearchCV
import time; start = time.time()

# BÚSQUEDA DE HIPERPARÁMETROS
clf = QuadraticDiscriminantAnalysis()
G = {'reg_param': loguniform(1e-4, 1e-1)} # reg_param: parámetro de regularizacion. Evita matrices de covarianzas singulares
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23) # 
RS = RandomizedSearchCV(clf, G, n_iter=10, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=32) # Prueba n_iter combinaciones aleatorias en paralelo (n_jobs = -1)

acc = RS.fit(X_train, y_train).score(X_test, y_test)

print(f"Precisión: {acc:.2%} con {RS.best_params_['reg_param']:.4f}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))


Precisión: 52.53% con 0.0008
Tiempo (hh:mm:ss): 00:05:13


In [13]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, ShuffleSplit, RandomizedSearchCV
from scipy.stats import loguniform; import time; start = time.time()
clf = LinearDiscriminantAnalysis(); G = {'tol': loguniform(1e-9, 1e-1)}
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23)
RS = RandomizedSearchCV(clf, G, n_iter=10, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=8)
acc = RS.fit(X_train, y_train).score(X_test, y_test)
print(f"Precisión: {acc:.2%} con {RS.best_params_['tol']:.4f}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

Precisión: 40.91% con 0.0129
Tiempo (hh:mm:ss): 00:00:21


<np>

# EJERCICIO EXAMEN

In [ ]:
"""2025_04_01_A1_cuaderno_turno1_3CO21.pdf

A lo largo de las sesiones de laboratorio, se ha llegado a alcanzar con el clasificador Naive Bayes Gaussiano
(NBG) un acierto del 81.5% en la tarea de MNIST. Con el objetivo de intentar mejorar esta tasa de acierto, completa el
siguiente cuaderno para realizar un experimento con NBG en el que se varíe la dimensionalidad de los datos mediante
PCA, y se haga una exploración del hiperparámetro var_smoothing . Completa la variable dni con las tres últimas
cifras de tu DNI sin letra para ser usada como semilla de inicialización en la exploración del hiperparámetro. Tras realizar
el experimento, comenta cuáles són los valores óptimos ( K y var_smoothing ) que has obtenido.


"""

# 1. Debemos realizar PCA para reducir la dimension de los datos a K
# 2. Con esos nuevos datos reducidos, realizamos la exploracion del hiperparametro var_smoothing
# 3. Hay que encontrar cuales son los valores optimos de K y var_smoothing para el experimento

import warnings

warnings.filterwarnings("ignore")
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, ShuffleSplit, RandomizedSearchCV
from sklearn.decomposition import PCA
from sklearn.naive_bayes import GaussianNB
from scipy.stats import loguniform

dni = 308


X, y = fetch_openml(data_id=554, return_X_y=True, as_frame=False, parser="liac-arff")
X /= 255.0
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=10000, shuffle=False
)

import time
start = time.time()
max_K = np.min(X_train.shape)
pca = PCA(n_components=max_K).fit(X_train) # Realizamos la reduccion de datos con PCA. No necesitamos retransformarlos a las dimensiones originales
Z_train = pca.transform(X_train)    # Obtenemos el caso completo. A partir de esto, "quitaremos" las ultimas K columnas en cada iteracion, hasta acabar usando todas
Z_test = pca.transform(X_test)

# PRIMERO: BÚSQUEDA DEL PARAMETRO K IDÓNEO:
Ks = np.array([50, 100, 200, 300, 400, 500, max_K])

# Como clf no depende de K, no tiene sentido que lo vayamos creando por cada iteracion
clf = GaussianNB()  # necesitamos el clasificador que tiene el parametro var_smoothing. Es GaussianNB
G = {"var_smoothing": loguniform(1e-9, 1e-1)}
splitter = ShuffleSplit(n_splits=1, test_size=0.1, random_state=333)
RS = RandomizedSearchCV(
    clf, G, n_iter=3, scoring="accuracy", n_jobs=-1, cv=splitter, pre_dispatch=32
)
for i, K in enumerate(Ks):
    # IMPORTANTE: EN LA BUSQUEDA DE HIPERPARAMETROS NO TENEMOS PORQUE VOLVER A LOS DATOS ORIGINALES. TRABAJAMOS CON LA PROYECCION Z_train y Z_test
    Z_train_K = Z_train[:, :K]
    Z_test_K = Z_test[:, :K]

    # Exploracion hiperparametro
    acc = RS.fit(Z_train_K, y_train).score(Z_test_K, y_test) # CRITICO: UTILIZAR Z_train_K y Z_test_K. 

    print(f"Precisión: {acc:.2%} con var_smoothing = {RS.best_params_['var_smoothing']:.4f} y K = {K}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

Precisión: 87.75% con var_smoothing = 0.0000 y K = 50
Precisión: 88.05% con var_smoothing = 0.0005 y K = 100
Precisión: 84.33% con var_smoothing = 0.0048 y K = 200
Precisión: 82.96% con var_smoothing = 0.0002 y K = 300
Precisión: 75.31% con var_smoothing = 0.0000 y K = 400
Precisión: 73.42% con var_smoothing = 0.0172 y K = 500
Precisión: 49.54% con var_smoothing = 0.0000 y K = 784
Tiempo (hh:mm:ss): 00:00:04
